# DVF Paris 2024–2025 — Merge Coordinates to Main Dataset
---

**Input 1:** `dvf_paris_2024_2025.csv`: main DVF dataset  
**Input 2:** `coordinate_matched_addresses.csv`: output of `address_to_coordinate_mapping_of_DVF_dataset.ipynb`  
**Output:** `dvf_paris_2024_2025_with_coordinates.csv`: full dataset with `lon`, `lat`, `matched_address`, `match_score` appended  

Merges on `address` (left join): row count of the main dataset is preserved.

## 1. Imports and Config

In [1]:
import pandas as pd

# Configuration
DVF         = "../data/dvf_paris_2024_2025.csv"
COORDINATES = "../data/coordinate_matched_addresses.csv"
OUTPUT      = "../data/dvf_paris_2024_2025_with_coordinates.csv"

# match_score threshold below which a match is flagged for review
LOW_SCORE_THRESHOLD = 0.5

Mounted at /content/drive


## 2. Load Data

In [3]:
df = pd.read_csv(DVF)

print("DVF dataset shape:", df.shape)
df.head(3)

DVF dataset shape: (156798, 20)


,transaction_number,transaction_date,transaction_type,property_value,street_number,street_type,street_code,street_name,postal_code,commune,department_code,commune_code,section,plot_number,lot_count,property_type_code,property_type,surface_area,room_count,year
0,1,2024-01-04,Sale,1042000.0,4.0,VLA,7288,PERREUR,75020,PARIS 20,75,120,BM,133,2,2.0,Apartment,86.0,4.0,2024
1,1,2024-01-04,Sale,1042000.0,4.0,VLA,7288,PERREUR,75020,PARIS 20,75,120,BM,133,2,3.0,Outbuilding,0.0,0.0,2024
2,1,2024-01-04,Sale,1042000.0,16.0,RUE,2786,DE LA DHUIS,75020,PARIS 20,75,120,BM,133,1,3.0,Outbuilding,0.0,0.0,2024


In [4]:
df_coords = pd.read_csv(COORDINATES)

print("Coordinates lookup shape:", df_coords.shape)
print("Columns:", list(df_coords.columns))
df_coords.head(3)

Coordinates lookup shape: (30696, 5)
Columns: ['address', 'lon', 'lat', 'matched_address', 'match_score']


,address,lon,lat,matched_address,match_score
0,4 VLA PERREUR 75020,2.404984,48.868239,4 Villa Perreur 75020 Paris,0.664930
1,16 RUE DE LA DHUIS 75020,2.404925,48.867963,16 Rue de la Dhuis 75020 Paris,0.819088
2,17 RUE DU ROI D ALGER 75018,2.347524,48.895295,17 Rue du Roi d'Alger 75018 Paris,0.823733


## 3. Rebuild "address" column

The merge key "address" must be rebuilt in the DVF dataset  
the same way it was done in: "address_to_coordinate_mapping_of_DVF_dataset.ipynb".  


"postal_code" is stored as float (75020.0). The .0 is stripped to match the values in "../data/coordinate_matched_addresses.csv".

In [5]:
# street_number is stored as float (e.g. 4.0).
# convert to string ("4") before concatenation
df["_street_num_str"] = df["street_number"].apply(
    lambda x: str(int(x)) if pd.notna(x) else ""
)

df["address"] = (
    df["_street_num_str"].str.strip() + " " +
    df["street_type"].str.strip()     + " " +
    df["street_name"].str.strip()     + " " +
    df["postal_code"].astype(str).str.replace(r"\.0$", "", regex=True).str.strip()
).str.strip()

# Drop helper column
df = df.drop(columns=["_street_num_str"])

print("Sample address values:")
print(df["address"].head(5).tolist())

Sample address values:
['4 VLA PERREUR 75020', '4 VLA PERREUR 75020', '16 RUE DE LA DHUIS 75020', '16 RUE DE LA DHUIS 75020', '4 VLA PERREUR 75020']


## 4. Merge Coordinates to Main Dataset

In [6]:
# Left join on address: preserves all rows from the main DVF dataset
df_with_coords = df.merge(df_coords, on="address", how="left")

print("Rows before merge:", len(df))
print("Rows after merge :", len(df_with_coords))
print("Row count match  :", len(df) == len(df_with_coords))
print()
print("New columns added:", ["address", "lon", "lat", "matched_address", "match_score"])

Rows before merge: 156798
Rows after merge : 156798
Row count match  : True

New columns added: ['address', 'lon', 'lat', 'matched_address', 'match_score']


In [7]:
# Preview original address columns and new columns
df_with_coords[
    ["street_number", "street_type", "street_name", "postal_code",
     "address", "lon", "lat", "matched_address", "match_score"]
].head(5)

,street_number,street_type,street_name,postal_code,address,lon,lat,matched_address,match_score
0,4.0,VLA,PERREUR,75020,4 VLA PERREUR 75020,2.404984,48.868239,4 Villa Perreur 75020 Paris,0.664930
1,4.0,VLA,PERREUR,75020,4 VLA PERREUR 75020,2.404984,48.868239,4 Villa Perreur 75020 Paris,0.664930
2,16.0,RUE,DE LA DHUIS,75020,16 RUE DE LA DHUIS 75020,2.404925,48.867963,16 Rue de la Dhuis 75020 Paris,0.819088
3,16.0,RUE,DE LA DHUIS,75020,16 RUE DE LA DHUIS 75020,2.404925,48.867963,16 Rue de la Dhuis 75020 Paris,0.819088
4,4.0,VLA,PERREUR,75020,4 VLA PERREUR 75020,2.404984,48.868239,4 Villa Perreur 75020 Paris,0.664930


## 5. Validation

In [8]:
# Check: how many rows received coordinates
total        = len(df_with_coords)
matched     = df_with_coords["lon"].notna().sum()
not_matched = df_with_coords["lon"].isna().sum()

print(f"Rows with coordinates    : {matched:,} ({matched/total:.1%})")
print(f"Rows without coordinates : {not_matched:,} ({not_matched/total:.1%})")
print()
print("match_score distribution:")
print(df_with_coords["match_score"].describe().round(3))

Rows with coordinates    : 151,516 (96.6%)
Rows without coordinates : 5,282 (3.4%)

match_score distribution:
count    151516.000
mean          0.771
std           0.084
min           0.312
25%           0.746
50%           0.812
75%           0.823
max           0.850
Name: match_score, dtype: float64


In [10]:
# Check whether rows with missing match scores are the same rows
# that also have missing coordinates. This helps confirm whether the
# missing score is simply caused by unsuccessful address matching

missing_score = df_with_coords["match_score"].isna()
missing_coords = df_with_coords["lon"].isna() | df_with_coords["lat"].isna()

print("Rows with missing match_score:", missing_score.sum())
print("Rows with missing coordinates:", missing_coords.sum())
print("Rows with both missing match_score and missing coordinates:", (missing_score & missing_coords).sum())

Rows with missing match_score: 5282
Rows with missing coordinates: 5282
Rows with both missing match_score and missing coordinates: 5282


In [11]:
# Flag low-confidence matches (<0.5) for manual review.
# We first filter all rows below the score threshold.
# Then we create a deduplicated review table so the same address match
# does not need to be checked multiple times.

low_confidence = df_with_coords[
    df_with_coords["match_score"].notna() &
    (df_with_coords["match_score"] < LOW_SCORE_THRESHOLD)
]

low_confidence_review = (
    low_confidence[["address", "matched_address", "match_score"]]
    .drop_duplicates()
    .sort_values(["match_score", "address"])
    .reset_index(drop=True)
)

if not low_confidence_review.empty:
    print()
    print(low_confidence_review.head(10).to_string(index=False))

print(f"All rows below score {LOW_SCORE_THRESHOLD}: {len(low_confidence):,}")
print(f"Unique matches below score {LOW_SCORE_THRESHOLD}: {len(low_confidence_review):,}")


                         address                                     matched_address  match_score
4 VC CHAUSSEE DE LA MUETTE 75016     4 Voie Communale Ham la Chaussée 60270 Gouvieux     0.312238
  17 VC PGE DE L INDUSTRIE 75010 Voie Communale Quai de l'Industrie 91200 Athis-Mons     0.329317
  20 VC PGE DE L INDUSTRIE 75010 Voie Communale Quai de l'Industrie 91200 Athis-Mons     0.329317
   9 VC PGE DE L INDUSTRIE 75010 Voie Communale Quai de l'Industrie 91200 Athis-Mons     0.335120
        53 BD SAINT MARCEL 75013                  53 Rue Brillat-Savarin 75013 Paris     0.343834
          5 IMP MONT LOUIS 75011                      5 Rue de Montreuil 75011 Paris     0.343891
        19 PL SAINT PIERRE 75018                        19 Rue Damrémont 75018 Paris     0.345687
            26 BD ST DENIS 75010              26 Boulevard de Strasbourg 75010 Paris     0.352685
         5 BD SAINT MARCEL 75013              5 Boulevard Vincent Auriol 75013 Paris     0.353369
         7 BD SAINT

In [ ]:
# Display the full table of unique low-confidence (< 0,5 match score) matches for manual review.

pd.set_option("display.max_rows", None)
display(low_confidence_review)

In [13]:
# Keep only rows with a valid match score of at least >= 0.5.

df_with_coords_cleaned = df_with_coords[
    df_with_coords["match_score"].notna() &
    (df_with_coords["match_score"] >= 0.5)
].copy()

print(f"Original rows: {len(df_with_coords):,}")
print(f"Cleaned rows: {len(df_with_coords_cleaned):,}")
print(f"Dropped rows: {len(df_with_coords) - len(df_with_coords_cleaned):,}")

Original rows: 156,798
Cleaned rows: 150,753
Dropped rows: 6,045


In [ ]:
# Review matches with scores between 0.5 and 0.6.
# These matches are not clearly low quality, but still uncertain enough
# to check obvious mismatches.

borderline_matches = df_with_coords[
    df_with_coords["match_score"].notna() &
    (df_with_coords["match_score"] >= 0.5) &
    (df_with_coords["match_score"] < 0.6)
]

borderline_review = (
    borderline_matches[["address", "matched_address", "match_score"]]
    .drop_duplicates()
    .sort_values(["match_score", "address"])
    .reset_index(drop=True)
)

print(f"All rows with score between 0.5 and 0.6: {len(borderline_matches):,}")
print(f"Unique matches with score between 0.5 and 0.6: {len(borderline_review):,}")

display(borderline_review)

In [15]:
# check: coordinates should fall within Paris
# Paris approx: lon 2.22–2.47, lat 48.81–48.91

out_of_bounds = df_with_coords_cleaned[
    df_with_coords_cleaned["lon"].notna() &
    df_with_coords_cleaned["lat"].notna() &
    (
        (df_with_coords_cleaned["lon"] < 2.22) |
        (df_with_coords_cleaned["lon"] > 2.47) |
        (df_with_coords_cleaned["lat"] < 48.81) |
        (df_with_coords_cleaned["lat"] > 48.91)
    )
]

print(f"Coordinates outside Paris: {len(out_of_bounds):,}")
if not out_of_bounds.empty:
    print(
        out_of_bounds[
            ["address", "matched_address", "lon", "lat", "match_score"]
        ]
        .drop_duplicates()
        .head(10)
        .to_string(index=False)
    )

Coordinates outside Paris: 24
                      address                           matched_address      lon       lat  match_score
 8 AV PIERRE 1ER SERBIE 75016  8 Rue Pierre 1er de Serbie 45000 Orléans 1.912964 47.908275     0.504178
18 AV PIERRE 1ER SERBIE 75016 18 Rue Pierre 1er de Serbie 45000 Orléans 1.911723 47.908625     0.514567


In [16]:
# Drop rows with coordinates outside the Paris.

df_cleaned_paris = df_with_coords_cleaned[
    df_with_coords_cleaned["lon"].isna() |
    df_with_coords_cleaned["lat"].isna() |
    (
        df_with_coords_cleaned["lon"].between(2.22, 2.47) &
        df_with_coords_cleaned["lat"].between(48.81, 48.91)
    )
].copy()

print(f"Original rows: {len(df_with_coords_cleaned):,}")
print(f"Cleaned rows: {len(df_cleaned_paris):,}")
print(f"Dropped rows outside Paris: {len(df_with_coords_cleaned) - len(df_cleaned_paris):,}")

Original rows: 150,753
Cleaned rows: 150,729
Dropped rows outside Paris: 24


## 6. Export

In [ ]:
df_cleaned_paris.to_csv(OUTPUT, index=False, encoding="utf-8")
print(f"Saved {len(df_cleaned_paris):,} rows to '{OUTPUT}'")
print(f"Columns: {list(df_with_coords.columns)}")